## Extractive Summarizer: SBERT Fine-Tuning + K-Means

Notebook này thực hiện:
1. **Fine-Tuning SBERT (Tiếng Anh & Tiếng Việt)** sử dụng tập cặp câu Oracle và `CosineSimilarityLoss`
3. **Đánh giá & Ablation Study**: So sánh Lead-3, TextRank, SBERT-No-KMeans và FineTuned-SBERT-KMeans trên các chỉ số ROUGE-1, ROUGE-2, ROUGE-L và Diversity Score.

In [ ]:
import torch
is_gpu = torch.cuda.is_available()
print(f"CUDA/GPU Available: {is_gpu}")
if is_gpu:
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"Tên GPU: {gpu_name}")
    print(f"Tổng dung lượng VRAM: {vram_gb:.2f} GB")

CUDA/GPU Available: True
Tên GPU: Tesla T4
Tổng dung lượng VRAM: 14.56 GB


## 1. Cài đặt Môi trường & Thư viện

In [ ]:
import os
import sys

if 'colab' in str(get_ipython()):
    if not os.path.exists('Extractive-Summarizer'):
        !git clone https://github.com/kttt294/Extractive-Summarizer.git
    if os.path.exists('Extractive-Summarizer') and not os.getcwd().endswith('Extractive-Summarizer'):
        %cd Extractive-Summarizer
    !git pull
    !pip install -q -r requirements.txt
sys.path.append(os.getcwd())

Cloning into 'Extractive-Summarizer'...
remote: Enumerating objects: 113, done.
remote: Counting objects: 100% (113/113), done.
remote: Compressing objects: 100% (88/88), done.
remote: Total 113 (delta 35), reused 96 (delta 20), pack-reused 0 (from 0)
Receiving objects: 100% (113/113), 14.15 MiB | 19.15 MiB/s, done.
Resolving deltas: 100% (35/35), done.
/content/Extractive-Summarizer
Already up to date.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 50.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 101.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 112.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━

## 2. Fine-Tuning Mô hình SBERT Song ngữ (English & Vietnamese)

**Cấu hình Siêu tham số:**
- `sample_data_count = 3000`: Lấy 3,000 bài báo (~12,000 cặp câu Oracle). Đạt điểm bão hòa hiệu năng cao nhất, tránh hiện tượng *Catastrophic Forgetting* và tối ưu hóa thời gian chạy trên Colab GPU T4.
- `epochs = 3`: Chuẩn mực tối ưu từ tác giả Sentence-Transformers để Fine-tune Bi-Encoder qua CosineSimilarityLoss.
- `batch_size = 32`: Tối ưu hiệu năng tính toán ma trận trên GPU NVIDIA T4.

In [ ]:
from src.train import train_finetune_sbert

# Fine-tuning Mô hình SBERT Tiếng Anh (CNN/DailyMail)
print("BẮT ĐẦU FINE-TUNE SBERT TIẾNG ANH (3,000 BÀI BÁO)")
model_en_path = train_finetune_sbert(lang='en', epochs=3, batch_size=32, sample_data_count=3000)

# Fine-tuning Mô hình SBERT Tiếng Việt (VietNews)
print("\n\nBẮT ĐẦU FINE-TUNE SBERT TIẾNG VIỆT (3,000 BÀI BÁO)")
model_vi_path = train_finetune_sbert(lang='vi', epochs=3, batch_size=32, sample_data_count=3000)

BẮT ĐẦU FINE-TUNE SBERT TIẾNG ANH (3,000 BÀI BÁO)
--> BẮT ĐẦU FINE-TUNING SBERT (EN)
Nạp mô hình SBERT Gốc: sentence-transformers/all-MiniLM-L6-v2


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

Đang nạp bộ dữ liệu mẫu Tiếng Việt dự phòng...
Đang tự động tạo các cặp câu Oracle cho Fine-Tuning (mục tiêu: 12000 cặp)...
Đã khởi tạo thành công 500 cặp câu huấn luyện Oracle.
Đang Fine-tune SBERT trong 3 epochs...


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fine-Tuning hoàn tất! Mô hình đã được lưu tại: /content/Extractive-Summarizer/models/finetuned_sbert_vi



In [ ]:
import shutil
shutil.make_archive('finetuned_sbert_en', 'zip', './models/finetuned_sbert_en')
shutil.make_archive('finetuned_sbert_vi', 'zip', './models/finetuned_sbert_vi')
if 'colab' in str(get_ipython()):
    from google.colab import files
    files.download('finetuned_sbert_en.zip')
    files.download('finetuned_sbert_vi.zip')
    print("Đã nén và tải weights về máy.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Đã nén và tải weights về máy.


## 3. Khung Đánh giá Thực nghiệm & Ablation Study
So sánh trực tiếp kết quả ROUGE-1, ROUGE-2, ROUGE-L và Diversity giữa 4 phương pháp:
1. **Lead-3 Baseline**
2. **TextRank Baseline**
3. **SBERT-No-KMeans** *(Direct Top-K - Ablation Study)*
4. **FineTuned-SBERT-KMeans** *(Mô hình Đề xuất)*

In [ ]:
# Chạy Đánh giá & Ablation Study trên 50 bài báo thử nghiệm

from src.evaluate import evaluate_framework
evaluate_framework(lang='vi', sample_count=50)

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'bkai-foundation-models/vietnews' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'bkai-foundation-models/vietnews' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.



  CHẠY KHUNG ĐÁNH GIÁ THỰC NGHIỆM (NGÔN NGỮ VI)
Đang nạp bộ dữ liệu thử nghiệm VI (số lượng=50)...


`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'vietnews' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'vietnews' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Đang nạp bộ dữ liệu mẫu Tiếng Việt dự phòng...
Đang nạp mô hình SBERT: ./models/finetuned_sbert_vi ...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


Mô hình                   | Silhouette | Diversity  | ROUGE-1    | ROUGE-2    | ROUGE-L   
Lead-3                    | 0.0000     | 0.0000     | 59.5745   % | 43.0108   % | 53.1915   %
TextRank                  | 0.0000     | 0.0000     | 59.5745   % | 43.0108   % | 53.1915   %
SBERT-No-KMeans           | 0.0000     | 0.0278     | 55.3846   % | 26.5625   % | 35.3846   %
FineTuned-SBERT-KMeans    | 0.0994     | 1.0000     | 72.0721   % | 45.8716   % | 50.4505   %



## 4. Demo Tóm tắt Văn bản Trực tiếp
Thử nghiệm trực tiếp pipeline tóm tắt trên một văn bản báo chí thực tế.

In [ ]:
from src.preprocess import resolve_language
from src.evaluate import run_sbert_pipeline

sample_article = '''
Trí tuệ nhân tạo đang thay đổi các ngành công nghiệp trên toàn cầu với tốc độ chưa từng có.
Những tiến bộ gần đây trong xử lý ngôn ngữ tự nhiên cho phép máy tính hiểu và tóm tắt các tài liệu phức tạp chỉ trong vài giây.
Kỹ thuật tóm tắt trích xuất chọn trực tiếp các câu chứa nhiều thông tin nhất từ văn bản gốc.
Bằng cách tận dụng vector nhúng Sentence-BERT, máy tính có thể nắm bắt mối quan hệ ngữ nghĩa sâu sắc giữa các câu.
Thuật toán K-Means giúp gom nhóm các khái niệm tương đồng, đảm bảo bản tóm tắt cuối cùng bao quát nhiều góc độ khác nhau.
Kỹ thuật lọc sau (Post-filtering) tiếp tục loại bỏ các thông tin trùng lặp, tạo ra bản tóm tắt ngắn gọn và giàu giá trị cho người đọc.
Mô hình kết hợp này đại diện cho một phương pháp tiếp cận mạnh mẽ cho các hệ thống tóm tắt văn bản tự động hiện đại.
'''

lang = resolve_language(sample_article, user_choice='auto')
summary_text, summary_sents, sil_score, div_score = run_sbert_pipeline(sample_article, lang=lang, use_finetuned=True)

print(f"Ngôn ngữ tự động nhận diện: {lang.upper()}")
print(f"Chỉ số Nội tại (Intrinsic): Silhouette Score = {sil_score:.4f} | Diversity Score = {div_score:.4f}")
print("\n--- KẾT QUẢ TÓM TẮT TRÍCH XUẤT ---")
for i, sent in enumerate(summary_sents, 1):
    print(f"{i}. {sent}")

Ngôn ngữ tự động nhận diện: VI
Chỉ số Nội tại (Intrinsic): Silhouette Score = 0.0999 | Diversity Score = 0.2101

--- KẾT QUẢ TÓM TẮT TRÍCH XUẤT ---
1. Trí tuệ nhân tạo đang thay đổi các ngành công nghiệp trên toàn cầu với tốc độ chưa từng có.
2. Kỹ thuật tóm tắt trích xuất chọn trực tiếp các câu chứa nhiều thông tin nhất từ văn bản gốc.
3. Thuật toán K-Means giúp gom nhóm các khái niệm tương đồng, đảm bảo bản tóm tắt cuối cùng bao quát nhiều góc độ khác nhau.
4. Mô hình kết hợp này đại diện cho một phương pháp tiếp cận mạnh mẽ cho các hệ thống tóm tắt văn bản tự động hiện đại.
